# Project Extension: Telco Customer Churn Predictive Pipeline

### Project Context 
Following a telco customer churn data analysis using SQL and Power BI, this notebook focuses on extending the project into a predictive machine learning asset. The primary objective is to move from reactive descriptive analytics to proactive risk forecasting, specifically identifying high-risk telecom customers before they churn so the business can deploy targeted retention campaigns.

### Transition from Business Intelligence (BI) to Machine Learning (ML)
Because a deep Exploratory Data Analysis (EDA) was already completely executed inside SQL and visualixed via Power BI, significant features of churn are thoroughly mapped (e.g., high-risk profiles include month-to-month contracts, paperless billing, and high monthly chargers with low tenure). 

To avoid redundant visualisation workflows in Python, this notebook leverages those verified insights directly to select high-impact features, bypassing traditional exploratory plotting and transitioning straight into pipeline engineering and predictive modeling.

### Notebook Roadmap
1. **Feature Engineering & Ordinal Mapping:** Hardcoding business bins and converting to numerical matrices.
2. **Pipeline Architecture:** Building a leak-free processing structure with `scikit-learn`.
3. **Model Selection & Tuning:** Evaluating Logistic Regression, Decision Tree, and Random Forest architectures via `GridSearchCV`.
4. **Threshold Optimisation:** Tuning classification decision boundaries to align with real business cost trade-offs.

In [110]:
#Importing libraries
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    confusion_matrix,
    recall_score,
    precision_score,
    f1_score,
    classification_report,
    roc_auc_score
)

from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder,
    OrdinalEncoder
)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier



In [119]:
#Importing data (note the data has already been cleaned and validated in SQL)
df = pd.read_csv("/Users/pc/Downloads/churndata.csv")

In [86]:
#converting boleaan features to binary
cols = ["paperless_billing", "dependents", "phone_service"]
df[cols] = df[cols].astype(int)

In [87]:
#droping less significant features for model
drop_col = ["customer_id", "total_charges", "churn", "contract_type", "tenure", 
            "gender", "senior_citizen", "partner", "monthly_charge_bin"]

df1 = df.drop(columns=drop_col)



To prepare our dataset for the modeling pipeline, we perform targeted feature transformations based on our previous operational insights:

* **Fixed-Value Binning:** `monthly_charges` is segmented into a 4-tier business framework (0-30, 30-60, 60-90, 90-120) using specific boundaries established during our BI analysis phase. Because these thresholds are completely hardcoded, this process avoids data leakage.
* **Manual Ordinal Encoding:** Since these 4 bins have a natural directional scale, we map them directly to a clean, ordered numerical sequence `[0, 1, 2, 3]`. This explicitly preserves the structural boundaries and risk progression for our tree-based algorithms.
* **Early Feature Stripping:** The raw, continuous `monthly_charges` column and non-predictive structural identifiers (like `CustomerID`) are dropped prior to splitting to keep the feature space lean.
* **Splitting Data:** Data is split into training and testing sets for model training.

In [89]:
#creating monthly charges bin
df1["monthly_charges_bin"] = pd.cut(
    df["monthly_charges"],
    bins=[0, 30, 60, 90, 120],
    labels=[0, 1, 2, 3])

df1["tenure_bin"].unique()


array(['Loyal (25+ months)', 'Mid (7-12 months)',
       'Established (13-24 months)', 'Very Early (0-1 months)',
       'Early (2-3 months)', 'Early-Mid (4-6 months)'], dtype=object)

In [90]:
#splitting data for model training
X = df1.drop(["monthly_charges","churn_binary"] , axis=1) 
Y = df1["churn_binary"]

X_train, X_test, Y_train, Y_test = train_test_split(
    X,  Y, test_size=0.2)


Colomn Trasnformer is utilised to preprocess data in a more efficient and systematic manner.

* **Categorical Columns** (`Contract`, `PaymentMethod`, `InternetService`): Transformed using `OneHotEncoder()` to maintain matrix efficiency.
* **Ordinal Encoder:** Converting `tenure` to numerical values with ranking.`monthly_charges` feature was already converted to numerical formatting during the Pandas step, it bypasses the encoders and passes safely through the process using the `remainder='passthrough'` setting.

In [91]:
ordinal_features = ["tenure_bin"]
categorical_features =  ["contract","payment_method", "monthly_charges_bin","multiple_lines",
                       "internet_service", "online_security", "online_backup",
                         "device_protection", "tech_support", "streaming_tv", "streaming_movies"]

transformer = ColumnTransformer(transformers=
      [("ord", OrdinalEncoder(categories=[['Loyal (25+ months)', 'Mid (7-12 months)',
       'Established (13-24 months)', 'Very Early (0-1 months)',
       'Early (2-3 months)', 'Early-Mid (4-6 months)']] ), ordinal_features ),
        ( "cat", OneHotEncoder(drop="first", handle_unknown="ignore"),categorical_features)],
    remainder="passthrough")

We evaluate three distinct classification algorithms to strike a balance between clear business interpretability and raw predictive power:
1. **Logistic Regression:** Serves as our linear baseline.
2. **Decision Tree:** Formulates non-linear multi-variable interactions.
3. **Random Forest:** An ensemble method utilizing bagging to minimize variance and combat overfitting.

We implement `GridSearchCV` combined with 5-fold cross-validation. This systematically hunts for the absolute optimal hyperparameter combinations (such as tree depth and estimator counts), evaluating candidates against the ROC_AUC score.

In [92]:
logistic_reg_pipeline = Pipeline([("transformer", transformer),("model", LogisticRegression( max_iter=1000))])

logistic_reg_params = {"model__C": [0.01, 0.1, 1, 10]}

logistic_reg_grid = GridSearchCV(estimator=logistic_reg_pipeline, 
                                 param_grid=logistic_reg_params, cv=5, scoring="roc_auc", n_jobs=-1)

logistic_reg_grid.fit(X_train, Y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('transformer',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('ord',
                                                                         OrdinalEncoder(categories=[['Loyal '
                                                                                                     '(25+ '
                                                                                                     'months)',
                                                                                                     'Mid '
                                                                                                     '(7-12 '
                                                                                                     'months)',
                                                                                                     'Established '
                                                                                                     '(13-24 '
                                                                                                     'months)',
                                                                                                     'Very '
                                                                                                     'Early '
                                                                                                     '(0-1 '
                                                                                                     'months)',
                                                                                                     'Early '
                                                                                                     '(2-3 '
                                                                                                     'months)',
                                                                                                     'Early-Mid '
                                                                                                     '(4-6 '
                                                                                                     'months)']]),
                                                                         ['tenure_bin']),
                                                                        ('cat',
                                                                         OneHotEncoder(drop='first',
                                                                                       handle_unknown='ignore'),
                                                                         ['contract',
                                                                          'payment_method',
                                                                          'monthly_charges_bin',
                                                                          'multiple_lines',
                                                                          'internet_service',
                                                                          'online_security',
                                                                          'online_backup',
                                                                          'device_protection',
                                                                          'tech_support',
                                                                          'streaming_tv',
                                                                          'streaming_movies'])])),
                                       ('model',
                                        LogisticRegression(max_iter=1000))]),
             n_jobs=-1, param_grid={'model__C': [0.01, 0.1, 1, 10]},
             scoring='roc_auc')

In [93]:
random_forest_pipeline = Pipeline([("transformer", transformer),("model", RandomForestClassifier(random_state=42))])

random_forest_params = {"model__n_estimators": [100, 200],
                       "model__max_depth": [5, 8, 12],
                       "model__min_samples_split": [2, 5], 
                       "model__min_samples_leaf": [1, 2]}

random_forest_grid = GridSearchCV(estimator=random_forest_pipeline, param_grid=random_forest_params
                                  , cv=5, scoring="roc_auc", n_jobs=-1)

random_forest_grid.fit(X_train, Y_train)


GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('transformer',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('ord',
                                                                         OrdinalEncoder(categories=[['Loyal '
                                                                                                     '(25+ '
                                                                                                     'months)',
                                                                                                     'Mid '
                                                                                                     '(7-12 '
                                                                                                     'months)',
                                                                                                     'Established '
                                                                                                     '(13-24 '
                                                                                                     'months)',
                                                                                                     'Very '
                                                                                                     'Early '
                                                                                                     '(0-1 '
                                                                                                     'months)',
                                                                                                     'Early '
                                                                                                     '(2-3 '
                                                                                                     'months)',
                                                                                                     'Early-Mid '
                                                                                                     '(4-6 '
                                                                                                     'months)']]),
                                                                         ['tenure_bin']),
                                                                        ('cat',
                                                                         OneHotEncoder(drop=...
                                                                          'multiple_lines',
                                                                          'internet_service',
                                                                          'online_security',
                                                                          'online_backup',
                                                                          'device_protection',
                                                                          'tech_support',
                                                                          'streaming_tv',
                                                                          'streaming_movies'])])),
                                       ('model',
                                        RandomForestClassifier(random_state=42))]),
             n_jobs=-1,
             param_grid={'model__max_depth': [5, 8, 12],
                         'model__min_samples_leaf': [1, 2],
                         'model__min_samples_split': [2, 5],
                         'model__n_estimators': [100, 200]},
             scoring='roc_auc')

In [94]:
decision_tree_pipeline = Pipeline([("transformer", transformer),("model", DecisionTreeClassifier(random_state=42))])

decision_tree_params = {"model__max_depth": [3, 5, 8, 12],
                        "model__min_samples_split": [2, 5, 10],
                        "model__min_samples_leaf": [1, 2, 4]}

decision_tree_grid = GridSearchCV(estimator=decision_tree_pipeline, param_grid=decision_tree_params, cv=5, 
                                  scoring="roc_auc",n_jobs=-1)

decision_tree_grid.fit(X_train, Y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('transformer',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('ord',
                                                                         OrdinalEncoder(categories=[['Loyal '
                                                                                                     '(25+ '
                                                                                                     'months)',
                                                                                                     'Mid '
                                                                                                     '(7-12 '
                                                                                                     'months)',
                                                                                                     'Established '
                                                                                                     '(13-24 '
                                                                                                     'months)',
                                                                                                     'Very '
                                                                                                     'Early '
                                                                                                     '(0-1 '
                                                                                                     'months)',
                                                                                                     'Early '
                                                                                                     '(2-3 '
                                                                                                     'months)',
                                                                                                     'Early-Mid '
                                                                                                     '(4-6 '
                                                                                                     'months)']]),
                                                                         ['tenure_bin']),
                                                                        ('cat',
                                                                         OneHotEncoder(drop=...
                                                                          'monthly_charges_bin',
                                                                          'multiple_lines',
                                                                          'internet_service',
                                                                          'online_security',
                                                                          'online_backup',
                                                                          'device_protection',
                                                                          'tech_support',
                                                                          'streaming_tv',
                                                                          'streaming_movies'])])),
                                       ('model',
                                        DecisionTreeClassifier(random_state=42))]),
             n_jobs=-1,
             param_grid={'model__max_depth': [3, 5, 8, 12],
                         'model__min_samples_leaf': [1, 2, 4],
                         'model__min_samples_split': [2, 5, 10]},
             scoring='roc_auc')

In [96]:
results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Decision Tree",
        "Random Forest"
    ],

    "Best ROC_AUC": [
        logistic_reg_grid.best_score_,
        decision_tree_grid.best_score_,
        random_forest_grid.best_score_
    ]
})
print(results)

                 Model  Best ROC_AUC
0  Logistic Regression      0.833163
1        Decision Tree      0.821092
2        Random Forest      0.835537


In [98]:
random_forest_grid.best_params_

{'model__max_depth': 8,
 'model__min_samples_leaf': 2,
 'model__min_samples_split': 2,
 'model__n_estimators': 200}

In [101]:
best_model = random_forest_grid.best_estimator_

In [112]:
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]
report_dict = classification_report(Y_test, y_pred, output_dict=True)
df_report = pd.DataFrame(report_dict).transpose()
df_report


,precision,recall,f1-score,support
0,0.837391,0.933140,0.882676,1032.000000
1,0.733591,0.503979,0.597484,377.000000
accuracy,0.818311,0.818311,0.818311,0.818311
macro avg,0.785491,0.718559,0.740080,1409.000000
weighted avg,0.809618,0.818311,0.806369,1409.000000


In [113]:
print("ROC AUC:",roc_auc_score(Y_test, y_prob))

ROC AUC: 0.8697270886023895


By default, classification algorithms separate "Churn" from "No Churn" at a standard 50% probability threshold ($0.5$). However, in a live telecom environment, prediction errors carry heavily asymmetric business costs:
* **False Negative (High Cost):** Predicting a customer will stay, but they leave. We lose their total lifetime contract revenue.
* **False Positive (Low Cost):** Predicting a customer will leave, but they stay. We mistakenly target them with a low-cost marketing incentive or discount.

Because missing an at-risk customer is significantly more damaging to the bottom line than over-incentivizing a loyal one, a $0.5$ threshold is financially suboptimal. 

To isolate the most cost-effective operating point, we execute a targeted evaluation loop across critical threshold values: `[0.2, 0.3, 0.4, 0.5]`. For each step in the loop, we generate:
1. **Confusion Matrices:** To track raw shifts in False Negatives vs. False Positives.
2. **Recall:** To verify the exact percentage of actual churners we successfully capture.
3. **Precision & F1-Scores:** To monitor our marketing budget efficiency and prevent over-saturation.

By analysing the resulting metrics side-by-side, we deliberately select an optimised business threshold that aggressively maximizes customer retention while ensuring campaign expenses remain viable.

In [116]:
thresholds = [0.2, 0.3, 0.4, 0.5]

for t in thresholds:
    
    y_pred = (y_prob >= t).astype(int)
    
    recall = recall_score(Y_test, y_pred)
    precision = precision_score(Y_test, y_pred)
    f1 = f1_score(Y_test, y_pred)
       
    print(f"\nThreshold: {t}")
    print(confusion_matrix(Y_test, y_pred))
    print(f"Recall: {recall:.3f}")
    print(f"Precision: {precision:.3f}")
    print(f"F1-score: {f1:.3f}")



Threshold: 0.2
[[635 397]
 [ 32 345]]
Recall: 0.915
Precision: 0.465
F1-score: 0.617

Threshold: 0.3
[[766 266]
 [ 65 312]]
Recall: 0.828
Precision: 0.540
F1-score: 0.653

Threshold: 0.4
[[880 152]
 [117 260]]
Recall: 0.690
Precision: 0.631
F1-score: 0.659

Threshold: 0.5
[[963  69]
 [187 190]]
Recall: 0.504
Precision: 0.734
F1-score: 0.597


As threshold goes from 0.2 to 0.5, the model becomes:
* more conservative 
* predicts churn less often 

From the results we observe a trend, as threshold increases:
* Recall	decreases
* Precision	increases
* False Positives	decrease
* False Negatives	increase

Depending on business goals I would probably choose a **threshold of 0.3** because it gives:
* strong recall (0.828) 
* acceptable precision (0.540) 
* good F1 balance 
* much fewer missed churners than 0.4 or 0.5 

Increasing the threshold misses nearly half the churners which defeats much of the purpose of churn prediction. Decreasing the threshold results in more than half your interventions are wasted (397 false positives at 0.2). 

